# Tennis Match History Project
**Team:** Phouvadeth Vathana, Joseph Bronyah, Dhrumil Patel

This notebook runs the code in `tennis_project.py` and `tennis_analyses.py`.
Keep both files in the same folder as this notebook.

**Questions:** performance after ranking upsets; repeat-opponent history;
performance against left-handed opponents.

The output is observational evidence, not proof of psychological recovery or causation.
Read `README.md` for the scope, match-order approximation, and model definitions.

## 1. Install dependencies
Run `python -m pip install -r requirements.txt` in your terminal before opening the notebook.
The cell below is an optional alternative for an existing notebook environment.

In [ ]:
# Uncomment if needed, then restart the kernel after installation.
# %pip install -r requirements.txt

In [ ]:
from pathlib import Path
import json
import logging
import pandas as pd
from IPython.display import display, Image
from tennis_project import Config, run_project

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
assert Path("tennis_analyses.py").exists(), "Open this notebook from the extracted project folder."

## 2. Choose the full study or the offline demonstration

`RUN_FULL_STUDY = True` downloads the pinned annual ATP files. The default main
period is 2000–2024, with 1990–1999 for earlier history. Change the configuration
before running to adjust the dates or upset definition.

`False` runs **31 real source rows from Brisbane 2024**. This checks code only;
it is not enough data to answer the research questions. The models will be skipped.

Default ranking upset: a top-50 player loses to someone at least 30 ranking
places lower. This is a chosen ranking-based definition, not betting odds.

In [ ]:
RUN_FULL_STUDY = True

config = Config(
    history_start=1990,
    analysis_start=2000,
    end_year=2024,
    test_start=2020,
    upset_max_rank=50,
    upset_rank_gap=30,
    min_hand_matches=10,
    bootstrap_samples=500,
    output_dir="outputs",
)
input_files = None
if not RUN_FULL_STUDY:
    config.history_start = config.analysis_start = config.end_year = 2024
    config.output_dir = "demo_outputs"
    input_files = [Path("tests/fixtures/brisbane_2024_excerpt.csv")]
    print("DEMONSTRATION ONLY: not the full research dataset.")

## 3. Download, validate, build features, and analyze
The program caches original files and records hashes. Failed downloads are not
silently skipped. A rerun replaces known generated outputs in the selected
output directory; use a new directory to preserve an earlier configuration.

In [ ]:
summary = run_project(config, input_files=input_files)
output = Path(config.output_dir)
print("Raw match rows:", summary["raw_match_rows"])
print("Eligible analysis matches:", summary["eligible_analysis_matches"])
for question in ["q1", "q2", "q3"]:
    print(question.upper(), summary[question]["status"])

## 4. Check data quality before interpreting results
Counts refer to physical matches. The feature table has two player views per
match, so its row count is not a count of independent matches.

The program screens incomplete results and ambiguity, but does not independently
verify exact match dates or every tennis score.

In [ ]:
display(pd.read_csv(output / "coverage_by_year.csv"))
display(pd.read_csv(output / "missing_values.csv"))
display(pd.read_csv(output / "player_row_quality.csv"))
print("Exclusion flags (overlapping):")
print(json.dumps(summary["exclusion_flag_counts_nonexclusive"], indent=2))

In [ ]:
features = pd.read_csv(output / "player_match_features.csv.gz")
columns = ["match_id", "player_name", "opponent_name", "won", "eligible",
           "player_age", "player_hand", "opponent_hand", "prior_matches",
           "recent_win_rate", "prior_h2h_meetings", "prior_h2h_wins"]
display(features[columns].head(12))

## 5. Question 1: performance after ranking upsets
Age and experience are separate variables. Experience counts earlier eligible
recorded matches, not the entire professional career.

A missing follow-up window is not zero wins. The next-three-match rate is
calculated only when all three next recorded matches are eligible. We do not
skip a retirement or an ambiguous event to find later favorable observations.

The following summaries are **unadjusted**. The optional adjusted rate model
uses a common rank-eligible population and accounts for observed strength,
recent form, age, experience, surfaces, format, and season. Its results are
exploratory associations, with focal-player-clustered uncertainty.

In [ ]:
display(pd.read_csv(output / "q1_summary_age.csv"))
display(pd.read_csv(output / "q1_summary_experience.csv"))
display(pd.read_csv(output / "q1_threshold_sensitivity.csv"))
print(json.dumps(summary["q1"], indent=2))

In [ ]:
q1_coefficients = output / "q1_adjusted_coefficients.csv"
if q1_coefficients.exists():
    display(pd.read_csv(q1_coefficients))
else:
    print("Q1 adjustment was not fitted. See the status above; no result has been invented.")

## 6. Question 2: do previous meetings improve predictions?
Compare the baseline, baseline plus meeting count, and baseline plus head-to-head
results. They use the same earlier-season training set and later-season test set.

**Lower Brier score and log loss are better.** In the paired Brier comparison,
a positive improvement means the full head-to-head model had lower error than
the baseline. An interval spanning zero does not provide a clear direction.

Earlier test outcomes can update later test histories, but the prediction
models remain frozen. No current-match statistics or future outcomes are predictors.
Opponent-pair bootstrap intervals do not capture all shared-player dependence.

In [ ]:
metrics_file = output / "q2_model_metrics.csv"
if metrics_file.exists():
    metrics = pd.read_csv(metrics_file)
    display(metrics.loc[metrics["meeting_group"].eq("all")])
    display(metrics.loc[~metrics["meeting_group"].eq("all")])
else:
    print("Insufficient data for the chronological model comparison.")
print(json.dumps(summary["q2"], indent=2))

## 7. Question 3: left-handed versus right-handed opponents
The paired table contains the **same player's** L-minus-R differences. Raw
service-point rates and win-rate differences are unadjusted and include counts.
A positive raw difference means better recorded performance against left-handers.

The optional within-player model adjusts match-result comparisons. Its
`opponent_left` coefficient describes right-handed focal players; add the
interaction for left-handed focal players. A coefficient of 0.02 means about
two percentage points, not 2% relative growth. This is not a causal estimate.

In [ ]:
by_hand = pd.read_csv(output / "q3_by_player_and_hand.csv")
paired = pd.read_csv(output / "q3_paired_player_comparison.csv")
display(by_hand.head(12))
display(paired.head(12))
coeff_file = output / "q3_adjusted_coefficients.csv"
if coeff_file.exists():
    coefficients = pd.read_csv(coeff_file)
    display(coefficients.loc[coefficients["term"].str.contains("opponent_left")])
else:
    print("Insufficient data for the within-player adjusted model.")
print(json.dumps(summary["q3"], indent=2))

## 8. Exploratory charts
The recovery and paired handedness charts are unadjusted descriptions. Read
the opportunity counts and adjusted outputs before interpreting them.

In [ ]:
chart_files = sorted((output / "charts").glob("*.png"))
if not chart_files:
    print("No charts available for this sample.")
for path in chart_files:
    display(Image(filename=str(path)))

## 9. Before writing the report
Report actual source and eligible match counts, missing information, exclusions,
and the chosen upset definition. Keep age and experience separate. Describe
match order as inferred from tournament week and knockout round, not verified
match-day dates. Label early-history counts as observed history rather than
complete careers. Report follow-up censoring and uncertainty.

`SOURCES.md` and `DATA_LICENSE.txt` contain attribution. `run_summary.json`
records configuration, model status, versions, and input hashes.

The bundled real sample and synthetic test fixtures verify implementation;
they do not establish the answers to your research questions.

In [ ]:
# Optional: run the automated implementation checks.
# !python -m pytest -q